In [0]:
# Databricks notebook (%python)
dbutils.widgets.text("fy_year_name", "FY2425")  # 例: FY2425
dbutils.widgets.text("fy_half_year_name", "2H") # 例: 2H

In [0]:
%sql
-- DSCからTarget取得 with Org
-- FY + Half をパラメータ化 + FN2置き換え（RTはcustomer_japan_dim_vw、WSはwarehouse_dim_vw）

WITH
params AS (
  SELECT
    '${fy_year_name}'      AS fy_year_name_param,
    '${fy_half_year_name}' AS fy_half_year_name_param
),

/* === 指定FY/Half の日付集合 === */
cal_fy_half AS (
  SELECT
    c.day_date,
    c.fy_quarter_name AS pg_quarter
  FROM hive_metastore.japan_gold.calender_dim_vw AS c
  CROSS JOIN params AS p
  WHERE c.fy_year_name = p.fy_year_name_param
    AND c.fy_half_year_name = p.fy_half_year_name_param
),

/* === パーティションプルーニング用にstart/endも作る === */
cal_bounds AS (
  SELECT
    MIN(day_date) AS start_ts,
    (MAX(day_date) + INTERVAL 1 DAY) AS end_ts_excl
  FROM cal_fy_half
),

/* === Org2-5 === */
org AS (
  SELECT
    o.org5_key,
    o.org2_name AS org2,
    o.org3_name AS org3,
    o.org4_name AS org4,
    o.org5_name AS org5
  FROM hive_metastore.japan_gold.sl_organization_dim_vw AS o
  WHERE o.org5_key IS NOT NULL
),

/* === GIV dimension（customer_org_mapping_key起点のマスタ） === */
giv_dim_base AS (
  SELECT
    gd.customer_org_mapping_key,
    gd.cust_key,
    gd.category_name AS category,
    gd.Org_Code      AS org_code,
    gd.org5_key,
    -- FN2欠損時のフォールバックとして保持
    gd.FN_Code       AS fn_code_current,
    gd.FN_Name       AS fn_name_current
  FROM hive_metastore.japan_gold.giv_dimension_dim_vw AS gd
),

/* === RT側の将来FN（FN2） === */
rt_cust AS (
  SELECT
    c.cust_key,
    c.jp_cust_parent_group_name AS parent_group_rt,
    c.jp_local_fn_code_2        AS fn2_code_rt,
    c.jp_local_fn_name_2        AS fn2_name_rt
  FROM hive_metastore.japan_gold.customer_japan_dim_vw AS c
),

/* === WS側の将来FN（FN2） === */
ws_cust AS (
  SELECT
    w.cust_key,
    w.jp_cust_parent_group_name AS parent_group_ws,
    w.jp_local_fn_code_2        AS fn2_code_ws,
    w.jp_local_fn_name_2        AS fn2_name_ws
  FROM hive_metastore.japan_gold.warehouse_dim_vw AS w
),

/* === cust_keyごとに「DC/WS」とFN2を確定（RT/WSで参照先を切替） === */
cust_fn2 AS (
  SELECT
    gdb.cust_key,
    COALESCE(wc.parent_group_ws, rc.parent_group_rt) AS dc_ws,

    -- NOTE: jp_cust_parent_group_name の値が 'WS' のときのみ卸側FN2を採用（値が違う場合は要調整）
    CASE
      WHEN COALESCE(wc.parent_group_ws, rc.parent_group_rt) = 'WS' THEN wc.fn2_code_ws
      ELSE rc.fn2_code_rt
    END AS fn_code_2,

    CASE
      WHEN COALESCE(wc.parent_group_ws, rc.parent_group_rt) = 'WS' THEN wc.fn2_name_ws
      ELSE rc.fn2_name_rt
    END AS fn_name_2
  FROM (SELECT DISTINCT cust_key FROM giv_dim_base) AS gdb
  LEFT JOIN ws_cust AS wc
    ON gdb.cust_key = wc.cust_key
  LEFT JOIN rt_cust AS rc
    ON gdb.cust_key = rc.cust_key
),

/* === Facts（日次で先に集計して行爆発を防ぐ） === */
ship_day AS (
  SELECT
    sh.customer_org_mapping_key,
    sh.day_date,
    SUM(sh.gross_transact_amt) AS shipment_giv_value
  FROM hive_metastore.japan_gold.shipments_fct_vw AS sh
  CROSS JOIN cal_bounds AS b
  WHERE sh.day_date >= b.start_ts
    AND sh.day_date <  b.end_ts_excl
  GROUP BY sh.customer_org_mapping_key, sh.day_date
),

indirect_day AS (
  SELECT
    isd.customer_org_mapping_key,
    isd.day_date,
    SUM(isd.gross_transact_amt) AS indirect_giv_shipments
  FROM hive_metastore.japan_linkedexcel.indirect_ship_day_fct_vw AS isd
  CROSS JOIN cal_bounds AS b
  WHERE isd.day_date >= b.start_ts
    AND isd.day_date <  b.end_ts_excl
  GROUP BY isd.customer_org_mapping_key, isd.day_date
),

garp_day AS (
  SELECT
    g.customer_org_mapping_key,
    g.day_date,
    SUM(g.garp_value) AS garp_value
  FROM hive_metastore.japan_gold.giv_garp_dim_vw AS g
  CROSS JOIN cal_bounds AS b
  WHERE g.day_date >= b.start_ts
    AND g.day_date <  b.end_ts_excl
  GROUP BY g.customer_org_mapping_key, g.day_date
),

outbound_day AS (
  SELECT
    go.customer_org_mapping_key,
    go.day_date,
    SUM(go.giv_outbound) AS giv_outbound
  FROM hive_metastore.japan_gold.giv_outbound_vw AS go
  CROSS JOIN cal_bounds AS b
  WHERE go.day_date >= b.start_ts
    AND go.day_date <  b.end_ts_excl
  GROUP BY go.customer_org_mapping_key, go.day_date
),

/* === 日付×mappingキーの母集合（どれかのfactに存在する日を拾う） === */
all_day_keys AS (
  SELECT customer_org_mapping_key, day_date FROM ship_day
  UNION
  SELECT customer_org_mapping_key, day_date FROM indirect_day
  UNION
  SELECT customer_org_mapping_key, day_date FROM garp_day
  UNION
  SELECT customer_org_mapping_key, day_date FROM outbound_day
),

/* === 日次ワイド化 === */
wide_day AS (
  SELECT
    k.customer_org_mapping_key,
    k.day_date,
    COALESCE(s.shipment_giv_value, 0.0)     AS shipment_giv_value,
    COALESCE(i.indirect_giv_shipments, 0.0) AS indirect_giv_shipments,
    COALESCE(g.garp_value, 0.0)             AS garp_value,
    COALESCE(o.giv_outbound, 0.0)           AS giv_outbound
  FROM all_day_keys AS k
  LEFT JOIN ship_day     AS s ON k.customer_org_mapping_key = s.customer_org_mapping_key AND k.day_date = s.day_date
  LEFT JOIN indirect_day AS i ON k.customer_org_mapping_key = i.customer_org_mapping_key AND k.day_date = i.day_date
  LEFT JOIN garp_day     AS g ON k.customer_org_mapping_key = g.customer_org_mapping_key AND k.day_date = g.day_date
  LEFT JOIN outbound_day AS o ON k.customer_org_mapping_key = o.customer_org_mapping_key AND k.day_date = o.day_date
)

/* === Final（PG Quarter集計 + FN2適用） === */
SELECT
  COALESCE(cf2.fn_code_2, gdb.fn_code_current) AS fn_code,
  COALESCE(cf2.fn_name_2, gdb.fn_name_current) AS fn_name,
  cf2.dc_ws AS dc_ws,
  gdb.category AS category,
  cal.pg_quarter AS pg_quarter,
  gdb.org_code AS org_code,
  org.org2 AS org2,
  org.org3 AS org3,
  org.org4 AS org4,
  org.org5 AS org5,

  SUM(w.shipment_giv_value) AS shipment_giv_value,
  SUM(w.indirect_giv_shipments) AS indirect_giv_shipments,
  SUM(w.shipment_giv_value + w.indirect_giv_shipments) AS ships_in_giv_customer_sales,
  SUM(w.garp_value) AS garp_value,
  SUM(w.giv_outbound) AS giv_outbound

FROM wide_day AS w
JOIN cal_fy_half AS cal
  ON w.day_date = cal.day_date
JOIN giv_dim_base AS gdb
  ON w.customer_org_mapping_key = gdb.customer_org_mapping_key
LEFT JOIN cust_fn2 AS cf2
  ON gdb.cust_key = cf2.cust_key
LEFT JOIN org
  ON gdb.org5_key = org.org5_key

GROUP BY
  COALESCE(cf2.fn_code_2, gdb.fn_code_current),
  COALESCE(cf2.fn_name_2, gdb.fn_name_current),
  cf2.dc_ws,
  gdb.category,
  cal.pg_quarter,
  gdb.org_code,
  org.org2, org.org3, org.org4, org.org5

ORDER BY
  fn_code,
  pg_quarter,
  category;